In [ ]:
project_path = "/home/jupyter"
import os
import sys
sys.path.append(project_path)
from google.cloud import bigquery, storage

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px

from fintrans_toolbox.src import bq_utils as bq

client = bigquery.Client()

In [ ]:
# Summarise the data by UK Cardholder Household Spending Online All Quarterly --------------- Cardholders' Number Total Quarterly 

UK_HH_Online = '''SELECT time_period_value, cardholders, destination_country, spend 
FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` 
where time_period = 'Quarter' 
and mcc = 'All' 
and mcg != 'All'
and mcg != 'BUSINESS TO BUSINESS' 
and merchant_channel = 'Online' 
and cardholder_origin_country = 'All' 
and cardholder_origin = 'UNITED KINGDOM' 
GROUP BY cardholders, destination_country, 
time_period_value, spend 
ORDER BY time_period_value, destination_country DESC'''
df_by_All = bq.read_bq_table_sql(client, UK_HH_Online)
df_by_All.head()

# Caculate UK Domestic Total Spending Quarterly

# Assuming df_by_All is the DataFrame returned from the BigQuery query
# Then group by 'time_period_value' and sum the 'spend' for each quarter

# Check if df_by_All is not None and has the expected columns
if df_by_All is not None and 'time_period_value' in df_by_All.columns and 'spend' in df_by_All.columns:
    # Group by quarter and sum the spend
    UK_HH_Online = df_by_All.groupby('time_period_value')['cardholders'].sum().reset_index()
   
 # Rename the column
    UK_HH_Online = UK_HH_Online.rename(columns={'cardholders': 'HH_Online_cardholders'})
    print(UK_HH_Online)
else:
    print("DataFrame is empty or missing required columns.")
    
# Save the result to a CSV file
csv_filename = "UK_HH_Online.csv"
UK_HH_Online.to_csv(csv_filename, index=False)

print(f"CSV file '{csv_filename}' has been created successfully.")


In [ ]:
# Total UK Cardholder Household Online Quarterly

# Summarise the data by UK Cardholder Online Household Spending Total Quarterly

UK_spending_HH_Online_All = '''SELECT time_period_value, SUM(spend) AS total_spend
FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` 
where time_period_value IN (
    '2019Q1', '2019Q2', '2019Q3', '2019Q4', '2020Q1', '2020Q2', '2020Q3', '2020Q4',
    '2021Q1', '2021Q2', '2021Q3', '2021Q4', '2022Q1', '2022Q2', '2022Q3', '2022Q4',
    '2023Q1', '2023Q2', '2023Q3', '2023Q4', '2024Q1', '2024Q2', '2024Q3', '2024Q4', '2025Q1'
  )
and mcg != 'All'
and mcg != 'BUSINESS TO BUSINESS' 
and merchant_channel = 'Online' 
and cardholder_origin_country = 'All' 
and cardholder_origin = 'UNITED KINGDOM' 
GROUP BY 
time_period_value 
ORDER BY time_period_value'''
df_by_HH_Online_All = bq.read_bq_table_sql(client, UK_spending_HH_Online_All)
df_by_HH_Online_All = df_by_HH_Online_All.rename(columns={'total_spend': 'Online_spend_HH'})
df_by_HH_Online_All.head()


# Save the DataFrame to a CSV file
csv_filename = "UK_HH_Online_Spending.csv"
df_by_HH_Online_All.to_csv(csv_filename, index=False)

print(f"CSV file '{csv_filename}' has been created successfully.")

In [ ]:
# UK Household Online Total Adjusted base value 2019Q1

import pandas as pd

# Load the data from the two CSV files
df_cardholders = pd.read_csv("UK_HH_Online.csv")
df_spend = pd.read_csv("UK_HH_Online_Spending.csv")

# Merge the two DataFrames on 'time_period_value'
df_merged = pd.merge(df_cardholders, df_spend, on="time_period_value", how="inner")

# Extract the 2019Q1 values
base_row = df_merged[df_merged["time_period_value"] == "2019Q1"]
if not base_row.empty:
    base_cardholders = base_row["HH_Online_cardholders"].values[0]
    base_spend = base_row["Online_spend_HH"].values[0]

    # Calculate adjusted quarterly spend
    df_merged["adjusted_Online_spend"] = (
        df_merged["HH_Online_cardholders"] / base_cardholders
    ) * base_spend

    # Save the result to a new CSV file
    df_merged.to_csv("Adjusted_Online_HH_Spend.csv", index=False)
    print("Adjusted quarterly spend saved to Adjusted_Online_HH_Spend.csv")
else:
    print("2019Q1 base data not found in the dataset.")

In [ ]:
# Summarise the data by UK Cardholder Household Spending F2F All Quarterly --------------- Cardholders' Number Total Quarterly 

UK_HH_F2F = '''SELECT time_period_value, cardholders, destination_country, spend 
FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` 
where time_period = 'Quarter' 
and mcc = 'All' 
and mcg != 'All'
and mcg != 'BUSINESS TO BUSINESS' 
and merchant_channel = 'Face to Face' 
and cardholder_origin_country = 'All' 
and cardholder_origin = 'UNITED KINGDOM' 
GROUP BY cardholders, destination_country, 
time_period_value, spend 
ORDER BY time_period_value, destination_country DESC'''
df_by_All = bq.read_bq_table_sql(client, UK_HH_F2F)
df_by_All.head()

# Caculate UK Domestic Total Spending Quarterly

# Assuming df_by_All is the DataFrame returned from the BigQuery query
# Then group by 'time_period_value' and sum the 'spend' for each quarter

# Check if df_by_All is not None and has the expected columns
if df_by_All is not None and 'time_period_value' in df_by_All.columns and 'spend' in df_by_All.columns:
    # Group by quarter and sum the spend
    UK_HH_F2F = df_by_All.groupby('time_period_value')['cardholders'].sum().reset_index()
   
 # Rename the column
    UK_HH_F2F = UK_HH_F2F.rename(columns={'cardholders': 'HH_F2F_cardholders'})
    print(UK_HH_F2F)
else:
    print("DataFrame is empty or missing required columns.")
    
# Save the result to a CSV file
csv_filename = "UK_HH_F2F.csv"
UK_HH_F2F.to_csv(csv_filename, index=False)

print(f"CSV file '{csv_filename}' has been created successfully.")


In [ ]:
# Total UK Cardholder Household F2F Quarterly

# Summarise the data by UK Cardholder F2F Household Spending Total Quarterly

UK_spending_HH_F2F_All = '''SELECT time_period_value, SUM(spend) AS total_spend
FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` 
where time_period_value IN (
    '2019Q1', '2019Q2', '2019Q3', '2019Q4', '2020Q1', '2020Q2', '2020Q3', '2020Q4',
    '2021Q1', '2021Q2', '2021Q3', '2021Q4', '2022Q1', '2022Q2', '2022Q3', '2022Q4',
    '2023Q1', '2023Q2', '2023Q3', '2023Q4', '2024Q1', '2024Q2', '2024Q3', '2024Q4', '2025Q1'
  )
and mcg != 'All'
and mcg != 'BUSINESS TO BUSINESS' 
and merchant_channel = 'Face to Face' 
and cardholder_origin_country = 'All' 
and cardholder_origin = 'UNITED KINGDOM' 
GROUP BY 
time_period_value 
ORDER BY time_period_value'''
df_by_HH_F2F_All = bq.read_bq_table_sql(client, UK_spending_HH_F2F_All)
df_by_HH_F2F_All = df_by_HH_F2F_All.rename(columns={'total_spend': 'F2F_spend_HH'})
df_by_HH_F2F_All.head()


# Save the DataFrame to a CSV file
csv_filename = "UK_HH_F2F_Spending.csv"
df_by_HH_F2F_All.to_csv(csv_filename, index=False)

print(f"CSV file '{csv_filename}' has been created successfully.")

In [ ]:
# UK Household F2F Total Adjusted base value 2019Q1

import pandas as pd

# Load the data from the two CSV files
df_cardholders = pd.read_csv("UK_HH_F2F.csv")
df_spend = pd.read_csv("UK_HH_F2F_Spending.csv")

# Merge the two DataFrames on 'time_period_value'
df_merged = pd.merge(df_cardholders, df_spend, on="time_period_value", how="inner")

# Extract the 2019Q1 values
base_row = df_merged[df_merged["time_period_value"] == "2019Q1"]
if not base_row.empty:
    base_cardholders = base_row["HH_F2F_cardholders"].values[0]
    base_spend = base_row["F2F_spend_HH"].values[0]

    # Calculate adjusted quarterly spend
    df_merged["adjusted_F2F_spend"] = (
        df_merged["HH_F2F_cardholders"] / base_cardholders
    ) * base_spend

    # Save the result to a new CSV file
    df_merged.to_csv("Adjusted_F2F_HH_Spend.csv", index=False)
    print("Adjusted quarterly spend saved to Adjusted_F2F_HH_Spend.csv")
else:
    print("2019Q1 base data not found in the dataset.")

In [ ]:
# Adjusted Indexed Spend Online & F2F
# Indexed card spending data (average 2019 equals 100) is calculated :
# Spend=(Adjusted Spend / Average Adjusted Spend in 2019) × 100

import pandas as pd

# Define a function to calculate Index Adjusted Spend
def calculate_index_adjusted_spend(df, spend_column, year_column='time_period_value', base_year=2019):
    # Filter the data for the base year
    base_year_data = df[df[year_column] == base_year]
    
    # Calculate the average adjusted spend for the base year
    average_base_year_spend = base_year_data[spend_column].mean()
    
    # Calculate the index adjusted spend
    df['Index_Adjusted_Spend'] = (df[spend_column] / average_base_year_spend) * 100
    
    return df

# Load the CSV files
f2f_df = pd.read_csv('Adjusted_F2F_HH_Spend.csv')
online_df = pd.read_csv('Adjusted_Online_HH_Spend.csv')

# Apply the function to both datasets
f2f_indexed = calculate_index_adjusted_spend(f2f_df, spend_column='adjusted_F2F_spend')
online_indexed = calculate_index_adjusted_spend(online_df, spend_column='adjusted_Online_spend')

# Save the results to new CSV files
f2f_indexed.to_csv('indexed_adjusted_F2F_spend.csv', index=False)
online_indexed.to_csv('indexed_adjusted_Online_spend.csv', index=False)

print("Indexed CSV files have been created successfully.")




In [ ]:
# Line chart for Online & F2F Adjusted Indexed Value

import pandas as pd
import matplotlib.pyplot as plt

# Load the indexed CSV files
f2f_file = "indexed_adjusted_F2F_spend.csv"
online_file = "indexed_adjusted_Online_spend.csv"

# Read the data
f2f_df = pd.read_csv(f2f_file)
online_df = pd.read_csv(online_file)

# Sort by time_period_value
f2f_df = f2f_df.sort_values("time_period_value")
online_df = online_df.sort_values("time_period_value")

# Merge the two datasets on time_period_value
merged_df = pd.merge(f2f_df, online_df, on="time_period_value", how="inner")

# Plot the indexed trends
plt.figure(figsize=(12, 6))
plt.plot(merged_df["time_period_value"], merged_df["adjusted_F2F_spend"], label="F2F Adjusted Spend Index", marker='o')
plt.plot(merged_df["time_period_value"], merged_df["adjusted_Online_spend"], label="Online Adjusted Spend Index", marker='o')
plt.xticks(rotation=45)
plt.xlabel("Quarter")
plt.ylabel("Index Adjusted Spend")
plt.title("Quarterly Index Adjusted Spend: F2F vs Online")
plt.legend()
plt.tight_layout()
plt.grid(True)
plt.savefig("Index_Adjusted_Spend_Comparison.png")
plt.show()
